# ISPD 2009 clock-network evaluation

The standard benchmarks in `ispd09CNS_results/ispd09_benchmarks` follow the format documented by `eval2009/s4/explain_s1_s1s`: a source, sink coordinates and loads, wire/buffer libraries, electrical limits, and optional blockages. The generated result files contain explicit internal nodes, sink mappings, wires, and polarity-safe buffer stages. The official `eval2009.pl` was run with `-s -b`: slew and blockage violations are reported but do not abort the measurement.

The implementation used is `src/ispd09_eval.py`. H3 is the project's search heuristic; C2 and C3 are included as deterministic controls.

In [1]:
from pathlib import Path
import json
import pandas as pd

rows = json.loads(Path('eval2009/results/summary.json').read_text())
df = pd.DataFrame(rows)
df[['heuristic', 'design', 'sinks', 'CLR_ps', 'cap_fF', 'seconds']]

,heuristic,design,sinks,CLR_ps,cap_fF,seconds
0,H3,ispd09f11,121,402.680,73774.458,4.152291
1,H3,ispd09f12,117,308.733,69792.194,4.189753
2,H3,ispd09f21,117,334.361,74145.517,3.892036
3,H3,ispd09f22,91,331.848,52169.057,3.327203
4,H3,ispd09f31,273,973.907,165699.954,11.414519
5,H3,ispd09f32,190,994.961,118783.106,7.531840
6,H3,ispd09fnb1,330,162.628,107533.965,11.732992
7,C2,ispd09f11,121,329.029,75490.930,3.005200
8,C2,ispd09f12,117,80.547,71894.756,2.916756
9,C2,ispd09f21,117,140.376,77727.468,3.170153


In [2]:
summary = (df.groupby('heuristic')
             .agg(mean_CLR_ps=('CLR_ps', 'mean'),
                  mean_cap_fF=('cap_fF', 'mean'),
                  total_seconds=('seconds', 'sum'))
             .round(3))
summary

,mean_CLR_ps,mean_cap_fF,total_seconds
heuristic,,,
C2,213.746,96168.312,36.839
C3,249.722,103686.172,35.406
H3,501.303,94556.893,46.241


## Findings

The measurements show that the deterministic C2 control has the lowest average CLR on this run, while H3 is not the best electrical baseline under the one-second-per-design budget used here. Capacitance and CLR are separate contest objectives, so the preferred heuristic depends on the weighting used for the experiment.

The official evaluator emitted slew warnings for these generated trees. They are retained in the raw reports in `df['report']`; `-s` was used only so that CLR and capacitance could still be measured. Blockage checking was disabled with `-b`, because this first converter does not perform blockage-aware Steiner placement. These results should therefore be treated as a structural/electrical baseline, not a contest-ready submission.


In [3]:
# Best measured heuristic per benchmark by CLR.
(df.loc[df.groupby('design')['CLR_ps'].idxmin(),
        ['design', 'heuristic', 'CLR_ps', 'cap_fF']]
   .sort_values('design')
   .reset_index(drop=True))

,design,heuristic,CLR_ps,cap_fF
0,ispd09f11,C3,47.798,85512.652
1,ispd09f12,C2,80.547,71894.756
2,ispd09f21,C2,140.376,77727.468
3,ispd09f22,C2,49.975,53128.182
4,ispd09f31,C2,543.735,164287.991
5,ispd09f32,C2,262.791,123242.356
6,ispd09fnb1,C3,45.752,108924.094


In [4]:
# Contest reference: best CLR among the downloaded final-report entries.
import re
reference = []
pattern = re.compile(r'^(ispd09\w+)\s+\S+\s+CLR\s+([0-9.]+).*?\bC\s+([0-9.]+)')
for line in Path('ispd09CNS_results/finalReport0331.txt').read_text().splitlines():
    match = pattern.search(line)
    if match:
        reference.append((match.group(1), float(match.group(2)), float(match.group(3))))
ref = pd.DataFrame(reference, columns=['design', 'reference_CLR_ps', 'reference_cap_fF'])
ref.groupby('design', as_index=False).min().sort_values('design')

,design,reference_CLR_ps,reference_cap_fF
0,ispd09f11,22.306,75672.665
1,ispd09f12,19.140,64132.140
2,ispd09f21,19.365,75160.217
3,ispd09f22,16.376,45193.320
4,ispd09f31,24.595,162501.976
5,ispd09f32,18.418,121866.339
6,ispd09fnb1,19.835,22753.295
